# PDPASA Source

In [3]:
from __future__ import annotations
import subprocess
import traceback
from pathlib import Path
import pandas as pd
import os
import textwrap
import urllib.request, zipfile, io


Load PDPASA step-horizon features.

In [4]:

def _fetch_pdpasa(run_start_str: str, run_end_str: str, forecasted_end_str: str):
    month = pd.Timestamp(run_start_str)

    if month >= pd.Timestamp("2024/08/01"):
        # Aug 2024+: AEMO renamed archive files; nemseer can't handle them, so download manually and call clean_forecast_csv() directly.
        cache_path = Path("Pre_processing/nemseer_pdpasa/extracted")
        year, m = month.year, str(month.month).zfill(2)
        old_name     = f"PUBLIC_DVD_PDPASA_REGIONSOLUTION_{year}{m}010000"
        csv_path     = cache_path / f"{old_name}.CSV"
        parquet_path = cache_path / f"{old_name}.parquet"

        if not csv_path.exists() and not parquet_path.exists():
            new_file = f"PUBLIC_ARCHIVE%23PDPASA_REGIONSOLUTION%23FILE01%23{year}{m}010000.zip"
            url = (
                f"http://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/"
                f"{year}/MMSDM_{year}_{m}/MMSDM_Historical_Data_SQLLoader/DATA/{new_file}"
            )
            print(f"  Downloading new-format archive for {year}-{m}...", flush=True)
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req) as resp:
                zip_data = resp.read()
            z = zipfile.ZipFile(io.BytesIO(zip_data))
            csvfiles = [n for n in z.namelist() if n.upper().endswith(".CSV")]
            if not csvfiles:
                raise RuntimeError(f"No CSV found in new-format archive for {year}-{m}")
            with open(csv_path, "wb") as f:
                f.write(z.read(csvfiles[0]))

        code = textwrap.dedent(f"""
            import pandas as pd
            from pathlib import Path
            from nemseer.data_handlers import clean_forecast_csv

            parquet_path = Path({str(parquet_path)!r})
            csv_path     = Path({str(csv_path)!r})

            if parquet_path.exists():
                df = pd.read_parquet(parquet_path)
            elif csv_path.exists():
                df = clean_forecast_csv(csv_path)
                df.to_parquet(parquet_path, index=False)
            else:
                raise FileNotFoundError(f"Expected CSV not found: {{csv_path}}")

            df.to_parquet({"Pre_processing/nemseer_pdpasa/temp.parquet"!r}, index=False)
        """)

    else:
        code = textwrap.dedent(f"""
            import nemseer, pandas as pd
            data = nemseer.compile_data(
                run_start={run_start_str!r},
                run_end={run_end_str!r},
                forecasted_start={run_start_str!r},
                forecasted_end={forecasted_end_str!r},
                raw_cache="Pre_processing/nemseer_pdpasa/extracted",
                processed_cache="Pre_processing/nemseer_pdpasa/processed",
                forecast_type="PDPASA",
                tables=["REGIONSOLUTION"],
            )
            if isinstance(data, dict):
                df = data.get("REGIONSOLUTION")
                if df is None:
                    df = data.get("regionsolution")
            else:
                df = data.to_dataframe().reset_index()
            df.to_parquet({"Pre_processing/nemseer_pdpasa/temp.parquet"!r}, index=False)
        """)

    result = subprocess.run(
        [r"C:\Users\danie\conda_envs\nemseer_env\python.exe", "-c", code],
        capture_output=True, text=True, cwd=Path.cwd(),
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"nemseer subprocess failed for {run_start_str[:7]} (exit {result.returncode}):\n"
            f"STDOUT:\n{result.stdout if result.stdout else '(none)'}\n"
            f"STDERR:\n{result.stderr if result.stderr else '(none)'}"
        )

    df = pd.read_parquet("Pre_processing/nemseer_pdpasa/temp.parquet")
    os.remove("Pre_processing/nemseer_pdpasa/temp.parquet")
    return df


def _cleanup_month_cache(month: pd.Timestamp) -> None:
    """Delete nemseer cache files for the given month from both extracted and processed dirs."""
    pattern = month.strftime("%Y%m")
    for cache_dir in [
        Path("Pre_processing/nemseer_pdpasa/extracted"),
        Path("Pre_processing/nemseer_pdpasa/processed"),
    ]:
        if cache_dir.exists():
            for f in cache_dir.glob(f"*{pattern}*"):
                if f.is_file():
                    f.unlink()


def main():
    data_start  = pd.Timestamp("2018/01/01")
    data_end    = pd.Timestamp("2026/01/01")

    fetch_start = data_start - pd.Timedelta(days=1)
    fetch_start_month = fetch_start.to_period("M").to_timestamp()
    total_months = pd.date_range(fetch_start_month, data_end - pd.Timedelta(days=1), freq="MS")

    # Determine which months have already been processed
    processed_path = Path("Processed_data/7_pdpasa.csv")
    already_processed = set()
    if processed_path.exists():
        existing_idx = pd.read_csv(processed_path, usecols=[0], parse_dates=True).iloc[:, 0]
        already_processed = set(pd.to_datetime(existing_idx).dt.to_period("M"))
        print(f"  Found {len(already_processed)} already-processed month(s) — will skip.", flush=True)

    monthly_slim = []

    for i, month in enumerate(total_months):
        # For the Dec 2017 pre-fetch month, use Jan 2018 as the proxy period to check
        check_period = max(month, data_start).to_period("M")
        if check_period in already_processed:
            print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} skipping (already processed).", flush=True)
            continue

        print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} fetching...", flush=True)

        next_month_4am = (month + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)).replace(hour=4, minute=0)
        month_end = next_month_4am if next_month_4am < data_end else data_end - pd.Timedelta(minutes=30)

        if month < pd.Timestamp("2024/08/01") and month_end >= pd.Timestamp("2024/08/01"):
            month_end = pd.Timestamp("2024/08/01") - pd.Timedelta(minutes=30)

        run_start_str = month.strftime("%Y/%m/%d %H:%M")
        run_end_str = month_end.strftime("%Y/%m/%d %H:%M")

        days_ahead = 2 if month_end.hour >= 13 else 1
        forecasted_end_str = (month_end + pd.Timedelta(days=days_ahead)).normalize() + pd.Timedelta(hours=4)
        forecasted_end_str = forecasted_end_str.strftime("%Y/%m/%d %H:%M")

        try:
            API_call_data = _fetch_pdpasa(run_start_str, run_end_str, forecasted_end_str)
        except RuntimeError as e:
            if not ("404" in str(e) or "Table(s) not available" in str(e)) or month >= pd.Timestamp("2024/08/01"):
                raise
            month_end_capped = (month + pd.offsets.MonthEnd(0)).replace(hour=23, minute=30)
            run_end_capped = month_end_capped.strftime("%Y/%m/%d %H:%M")
            days_ahead_capped = 2 if month_end_capped.hour >= 13 else 1
            forecasted_end_capped = (
                (month_end_capped + pd.Timedelta(days=days_ahead_capped)).normalize()
                + pd.Timedelta(hours=4)
            ).strftime("%Y/%m/%d %H:%M")
            try:
                print(f"  WARNING: next-month archive gap, retrying within {month:%Y-%m} only...", flush=True)
                API_call_data = _fetch_pdpasa(run_start_str, run_end_capped, forecasted_end_capped)
            except RuntimeError as e2:
                if "404" in str(e2) or "Table(s) not available" in str(e2):
                    print(f"  WARNING: AEMO archive missing for {month:%Y-%m}, skipping month.", flush=True)
                    continue
                raise

        # Slim to long-format (run_time, col, value) immediately
        raw = API_call_data.copy()
        raw.columns = [str(c).upper() for c in raw.columns]
        run_time      = pd.to_datetime(raw["RUN_DATETIME"], errors="coerce")
        forecast_time = pd.to_datetime(raw["INTERVAL_DATETIME"], errors="coerce")
        avail         = pd.to_numeric(raw["AGGREGATEPASAAVAILABILITY"], errors="coerce")
        demand50      = pd.to_numeric(raw["DEMAND50"], errors="coerce")
        reservecond   = pd.to_numeric(raw["RESERVECONDITION"], errors="coerce")
        horizon_step  = ((forecast_time - run_time).dt.total_seconds() / 1800.0).round().astype(int)
        region        = raw["REGIONID"].str.lower().str.strip()

        slim = pd.DataFrame({
            "run_time":     run_time,
            "region":       region,
            "horizon_step": horizon_step,
            "avail":        avail,
            "demand50":     demand50,
            "reservecond":  reservecond,
        }).dropna(subset=["run_time", "region", "horizon_step"])

        monthly_slim.append(slim)
        del API_call_data, raw, slim
        _cleanup_month_cache(month)
        print(f"  {month:%Y-%m} cache cleaned.", flush=True)

    if not monthly_slim:
        print("No new months to process.", flush=True)
        return pd.read_csv(processed_path, index_col=0, parse_dates=True).tail() if processed_path.exists() else None

    all_slim = pd.concat(monthly_slim, ignore_index=True)
    del monthly_slim

    # Pivot: index=run_time, columns=(region, metric, horizon_step)
    all_slim["col"] = (
        all_slim["region"] + "_"
        + all_slim[["avail","demand50","reservecond"]].stack().reset_index(level=1, drop=True).groupby(level=0).transform(lambda x: x.name if hasattr(x, "name") else "val")
    )

    avail_piv    = all_slim.pivot_table(index="run_time", columns=["region","horizon_step"], values="avail",    aggfunc="mean")
    demand50_piv = all_slim.pivot_table(index="run_time", columns=["region","horizon_step"], values="demand50", aggfunc="mean")
    reserv_piv   = all_slim.pivot_table(index="run_time", columns=["region","horizon_step"], values="reservecond", aggfunc="mean")

    avail_piv.columns    = [f"{r}_avail_h{h}"    for r,h in avail_piv.columns]
    demand50_piv.columns = [f"{r}_demand50_h{h}" for r,h in demand50_piv.columns]
    reserv_piv.columns   = [f"{r}_reservecond_h{h}" for r,h in reserv_piv.columns]

    df = pd.concat([avail_piv, demand50_piv, reserv_piv], axis=1).sort_index()
    df = df[~df.index.duplicated(keep="last")]
    del all_slim, avail_piv, demand50_piv, reserv_piv

    if processed_path.exists():
        existing = pd.read_csv(processed_path, index_col=0, parse_dates=True)
        existing.index = pd.to_datetime(existing.index, format="mixed")
        df = pd.concat([existing, df])
        df = df[~df.index.duplicated(keep="last")].sort_index()

    df.index.name = "run_time"
    df.to_csv(processed_path)
    return df.tail()

main()


  Found 96 already-processed month(s) — will skip.
  1/97 2017-12 skipping (already processed).
  2/97 2018-01 skipping (already processed).
  3/97 2018-02 skipping (already processed).
  4/97 2018-03 skipping (already processed).
  5/97 2018-04 skipping (already processed).
  6/97 2018-05 skipping (already processed).
  7/97 2018-06 skipping (already processed).
  8/97 2018-07 skipping (already processed).
  9/97 2018-08 skipping (already processed).
 10/97 2018-09 skipping (already processed).
 11/97 2018-10 skipping (already processed).
 12/97 2018-11 skipping (already processed).
 13/97 2018-12 skipping (already processed).
 14/97 2019-01 skipping (already processed).
 15/97 2019-02 skipping (already processed).
 16/97 2019-03 skipping (already processed).
 17/97 2019-04 skipping (already processed).
 18/97 2019-05 skipping (already processed).
 19/97 2019-06 skipping (already processed).
 20/97 2019-07 skipping (already processed).
 21/97 2019-08 skipping (already processed).
 22/

,pdpasa_avail_h1_nsw1,pdpasa_avail_h2_nsw1,pdpasa_avail_h3_nsw1,pdpasa_avail_h4_nsw1,pdpasa_avail_h5_nsw1,pdpasa_avail_h6_nsw1,pdpasa_avail_h7_nsw1,pdpasa_avail_h8_nsw1,pdpasa_avail_h9_nsw1,pdpasa_avail_h10_nsw1,...,pdpasa_reservecond_h69_vic1,pdpasa_reservecond_h70_vic1,pdpasa_reservecond_h71_vic1,pdpasa_reservecond_h72_vic1,pdpasa_reservecond_h73_vic1,pdpasa_reservecond_h74_vic1,pdpasa_reservecond_h75_vic1,pdpasa_reservecond_h76_vic1,pdpasa_reservecond_h77_vic1,pdpasa_reservecond_h78_vic1
SETTLEMENTDATE,,,,,,,,,,,,,,,,,,,,,
2025-12-31 21:30:00,13415.0,13491.0,13527.0,13546.0,13552.0,13557.0,13544.0,13505.0,13470.0,13449.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-31 22:00:00,13623.0,13648.0,13635.0,13633.0,13608.0,13565.0,13521.0,13481.0,13443.0,13425.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-31 22:30:00,13664.0,13646.0,13618.0,13528.0,13480.0,13431.0,13407.0,13402.0,13385.0,13354.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-31 23:00:00,13598.0,13571.0,13519.0,13472.0,13434.0,13417.0,13389.0,13366.0,13349.0,13329.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-31 23:30:00,13619.0,13553.0,13526.0,13495.0,13459.0,13435.0,13411.0,13364.0,13342.0,13402.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
